In [1]:
import sys
import os
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.metrics import classification_report

# Garante acesso ao módulo src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# 1. Conecta o código ao seu arquivo de banco de dados
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# 2. Define o nome do projeto (para não ficar tudo no 'Default')
mlflow.set_experiment("Fitness_System_Evolution")

from src.data_utils import get_preprocessing_pipeline


In [2]:
with mlflow.start_run(run_name="Baseline_Perceptron_Sistematico"):
    # Carregamento
    df = pd.read_csv('../data/fitness_dataset_linear.csv')
    X = df.drop('is_fit', axis=1)
    y = df['is_fit']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Pipeline Modular
    preprocessor = get_preprocessing_pipeline()
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', Perceptron(random_state=42))
    ])

    # Treino e Registo
    pipeline.fit(X_train, y_train)
    acc = pipeline.score(X_test, y_test)
    
    # Logs Automáticos
    mlflow.log_param("model_type", "Perceptron")
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(pipeline, "model_v1")
    
    print(f"Modelo Baseline registado no MLflow com {acc:.2%} de acurácia.")

2026/04/19 15:25:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:25:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Modelo Baseline registado no MLflow com 98.67% de acurácia.


In [3]:
# --- PARTE 2: INGESTÃO E AMOSTRAGEM ---

# Caminho relativo para garantir portabilidade
DATA_PATH = os.path.join('..', 'data', 'fitness_dataset_linear.csv')

def load_and_split_data(path):
    # Ingestão de dados
    df = pd.read_csv(path)
    
    # Separação de features e target
    X = df.drop('is_fit', axis=1)
    y = df['is_fit']
    
    # Estratégia de Amostragem: Hold-out (70/30)
    # O random_state=42 garante que a amostragem seja determinística (reprodutível)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    return X_train, X_test, y_train, y_test

# Executando a ingestão sistemática
X_train, X_test, y_train, y_test = load_and_split_data(DATA_PATH)

print(f"Ingestão concluída com sucesso.")
print(f"Amostras de Treino: {X_train.shape[0]} | Amostras de Teste: {X_test.shape[0]}")

Ingestão concluída com sucesso.
Amostras de Treino: 1400 | Amostras de Teste: 600


In [4]:
def data_diagnostic(X, y):
    print("--- DIAGNÓSTICO DE ENGENHARIA ---")
    
    # 1. Valores Ausentes
    nulls = X.isnull().sum()
    print(f"\n[Qualidade] Valores nulos encontrados:\n{nulls[nulls > 0] if nulls.any() else 'Nenhum'}")
    
    # 2. Inconsistências (Outliers rápidos)
    # Verificando se há idades negativas ou batimentos impossíveis
    if (X['age'] < 0).any() or (X['heart_rate'] < 30).any():
        print("\n[Alerta] Inconsistências detectadas em variáveis biométricas!")
    else:
        print("\n[Qualidade] Limites biométricos iniciais parecem consistentes.")
        
    # 3. Viés de Classe
    bias = y.value_counts(normalize=True) * 100
    print(f"\n[Viés] Proporção das classes no Target:\n{bias}")

# Chama o diagnóstico com os dados ingeridos
data_diagnostic(X_train, y_train)

--- DIAGNÓSTICO DE ENGENHARIA ---

[Qualidade] Valores nulos encontrados:
sleep_hours    118
dtype: int64

[Qualidade] Limites biométricos iniciais parecem consistentes.

[Viés] Proporção das classes no Target:
is_fit
0    50.0
1    50.0
Name: proportion, dtype: float64


In [5]:
#Experimento A: Baseline (Perceptron)
with mlflow.start_run(run_name="Exp_01_Perceptron_Baseline"):
    # Construção do pipeline completo
    model_perceptron = Pipeline(steps=[
        ('preprocessor', get_preprocessing_pipeline()),
        ('classifier', Perceptron(random_state=42))
    ])
    
    model_perceptron.fit(X_train, y_train)
    acc = model_perceptron.score(X_test, y_test)
    
    # Registro de metadados de engenharia
    mlflow.log_param("model_family", "linear")
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model_perceptron, "model_perceptron")
    print(f"Perceptron Finalizado: {acc:.4f}")

2026/04/19 15:25:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:25:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Perceptron Finalizado: 0.9850


In [6]:
#Experimento B: Árvore de Decisão com GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

with mlflow.start_run(run_name="Exp_02_DecisionTree_Otimizada"):
    pipeline_dt = Pipeline(steps=[
        ('preprocessor', get_preprocessing_pipeline()),
        ('classifier', DecisionTreeClassifier(random_state=42))
    ])
    
    param_grid = {
        'classifier__max_depth': [5, 10, 15],
        'classifier__criterion': ['gini', 'entropy']
    }
    
    # Validação Cruzada (CV=5) conforme enunciado
    grid_search = GridSearchCV(pipeline_dt, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    
    # Log de melhores parâmetros encontrados
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_accuracy", grid_search.best_score_)
    mlflow.log_metric("test_accuracy", grid_search.score(X_test, y_test))
    
    mlflow.sklearn.log_model(grid_search.best_estimator_, "model_dt_optimized")
    print("Decision Tree Otimizada e Registrada.")

2026/04/19 15:25:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:25:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree Otimizada e Registrada.


In [7]:
#Experimento C: Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

with mlflow.start_run(run_name="Exp_03_RandomForest_Complex"):
    pipeline_rf = Pipeline(steps=[
        ('preprocessor', get_preprocessing_pipeline()),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    
    pipeline_rf.fit(X_train, y_train)
    acc_rf = pipeline_rf.score(X_test, y_test)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", acc_rf)
    mlflow.sklearn.log_model(pipeline_rf, "model_rf")
    print(f"Random Forest Registrada: {acc_rf:.4f}")

2026/04/19 15:26:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:26:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Registrada: 0.9383


In [8]:
#PCA
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# --- EXPERIMENTO 04: PCA (Redução para 3 componentes) ---
with mlflow.start_run(run_name="Exp_04_PCA_Perceptron"):
    pipe_pca = Pipeline(steps=[
        ('preprocessor', get_preprocessing_pipeline()),
        ('pca', PCA(n_components=3)), # Reduzimos de 10 para 3 variáveis
        ('classifier', Perceptron(random_state=42))
    ])
    
    pipe_pca.fit(X_train, y_train)
    acc_pca = pipe_pca.score(X_test, y_test)
    
    mlflow.log_param("reduction_method", "PCA")
    mlflow.log_param("n_components", 3)
    mlflow.log_metric("accuracy", acc_pca)
    mlflow.sklearn.log_model(pipe_pca, "model_pca")
    print(f"PCA (3 comps) Finalizado. Acurácia: {acc_pca:.4f}")

# --- EXPERIMENTO 05: LDA (Redução para 1 componente) ---
with mlflow.start_run(run_name="Exp_05_LDA_Perceptron"):
    # Nota: LDA precisa das labels no fit da transformação
    pipe_lda = Pipeline(steps=[
        ('preprocessor', get_preprocessing_pipeline()),
        ('lda', LDA(n_components=1)), # LDA para binário é sempre 1
        ('classifier', Perceptron(random_state=42))
    ])
    
    pipe_lda.fit(X_train, y_train)
    acc_lda = pipe_lda.score(X_test, y_test)
    
    mlflow.log_param("reduction_method", "LDA")
    mlflow.log_metric("accuracy", acc_lda)
    mlflow.sklearn.log_model(pipe_lda, "model_lda")
    print(f"LDA Finalizado. Acurácia: {acc_lda:.4f}")

2026/04/19 15:26:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:26:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


PCA (3 comps) Finalizado. Acurácia: 0.6717


2026/04/19 15:26:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/19 15:26:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LDA Finalizado. Acurácia: 0.9683


In [9]:
import mlflow.sklearn
import pandas as pd

# 1. Busca o ID do experimento pelo nome
experiment = mlflow.get_experiment_by_name("Fitness_System_Evolution")
experiment_id = experiment.experiment_id

# 2. Busca a melhor "Run" (neste caso, a do LDA que teve ~96% de acurácia)
# Ordenamos pela métrica 'accuracy' de forma descendente
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.mlflow.runName = 'Exp_05_LDA_Perceptron'",
    order_by=["metrics.accuracy DESC"]
)

if not runs.empty:
    best_run_id = runs.iloc[0].run_id
    model_uri = f"runs:/{best_run_id}/model_lda"
    
    # 3. Carrega o modelo como se estivesse em Produção
    loaded_model = mlflow.sklearn.load_model(model_uri)
    
    # 4. Simulação de Inferência com um novo dado
    sample_data = X_test.iloc[:1]
    prediction = loaded_model.predict(sample_data)
    
    print(f"--- SIMULAÇÃO DE PRODUÇÃO ---")
    print(f"Modelo carregado da Run: {best_run_id}")
    print(f"Predição para o novo usuário: {'Condicionado' if prediction[0] == 1 else 'Não Condicionado'}")
else:
    print("Erro: Experimento LDA não encontrado no MLflow.")

--- SIMULAÇÃO DE PRODUÇÃO ---
Modelo carregado da Run: bf65f651eaea4c308789dc1a7db6e76c
Predição para o novo usuário: Não Condicionado
